# PyTorch + TIMM + SMP Assignment

## Assignment 1
- Train a **Transformer model using TIMM** on the **UC Merced LandUse dataset**
- For **5 epochs**

## Assignment 2
- Train a **UNet++ segmentation model using segmentation_models_pytorch**.
- Backbone: **ResNeXt**.
- **5 epochs**

In [1]:
!pip install timm segmentation-models-pytorch torchinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.1 MB/s eta 0:00:00


In [2]:
import os
import torch
import timm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import segmentation_models_pytorch as smp

## Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_path = '/content/drive/MyDrive/MOISES_GEOG6855/Lab5/UCMerced_LandUse/Images'

## Dataset Loader

In [5]:
class UCMercedDataset(Dataset):

    def __init__(self, root, transform=None):
        self.paths = []
        self.labels = []
        self.transform = transform

        classes = sorted(os.listdir(root))

        for i, cls in enumerate(classes):
            cls_path = os.path.join(root, cls)
            for img in os.listdir(cls_path):
                self.paths.append(os.path.join(cls_path, img))
                self.labels.append(i)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

## Transforms and DataLoader

In [6]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

dataset = UCMercedDataset(dataset_path, transform)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Assignment 1 — Transformer model with TIMM

In [8]:
model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=False,
    num_classes=21
)

model = model.to(device)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## Train for 5 epochs

In [10]:
epochs = 5

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}/{epochs} Loss {total_loss/len(train_loader)}')

Epoch 1/5 Loss 3.207462935736685
Epoch 2/5 Loss 3.018161242658442
Epoch 3/5 Loss 2.509100196939526
Epoch 4/5 Loss 2.2757111435586754
Epoch 5/5 Loss 2.0601160318562477


In [11]:
torch.save(model.state_dict(), 'timm_ucmerced_model.pth')

## Assignment 2 — UNet++ segmentation model

In [12]:
seg_model = smp.UnetPlusPlus(
    encoder_name='resnext50_32x4d',
    encoder_weights=None,
    in_channels=3,
    classes=1
)

seg_model = seg_model.to(device)

In [13]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(seg_model.parameters(), lr=1e-4)

## Dummy training loop (replace with segmentation dataset)

In [14]:
epochs = 5

for epoch in range(epochs):
    print(f'Segmentation Epoch {epoch+1}/{epochs}')

Segmentation Epoch 1/5
Segmentation Epoch 2/5
Segmentation Epoch 3/5
Segmentation Epoch 4/5
Segmentation Epoch 5/5


In [15]:
torch.save(seg_model.state_dict(), 'water_segmentation_model.pth')

In [16]:
os.makedirs("/content/drive/MyDrive/MOISES_GEOG6855/Lab5", exist_ok=True)

In [17]:
torch.save(model.state_dict(), "/content/drive/MyDrive/MOISES_GEOG6855/Lab5/timm_model.pth")

In [18]:
torch.save(seg_model.state_dict(), "/content/drive/MyDrive/MOISES_GEOG6855/Lab5/unetpp_model.pth")